# 03 · Agregación, reconocimiento y elección del modelo

IFRS 17 mide grupos, no pólizas. Un grupo reúne contratos de un mismo portafolio (riesgos similares gestionados juntos, par. 14), emitidos con no más de un año de diferencia (par. 22) y con un mismo nivel de rentabilidad al reconocerlos (par. 16):

- **ONE:** onerosos al reconocerlos (par. 47);
- **SIG:** sin posibilidad significativa de volverse onerosos (par. 16(b), 19);
- **RES:** el resto.

En los portafolios GMM se clasifican celdas homogéneas (plazo, banda de edad y canal) reconocidas cada mes, porque hay información razonable de que toda la celda cae en el mismo grupo (par. 17). Los supuestos son los vigentes al reconocer (los del cierre anterior), la curva es la de fin del mes de emisión y el ajuste por riesgo es el costo de capital. Una celda es SIG si sigue siendo rentable con mortalidad +15%, gastos +10% y caducidad ±30%.

In [1]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.float_format', lambda x: f'{x:,.0f}' if abs(x) >= 100 else f'{x:,.4f}')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
from ifrs17 import closing as C, pc
res = C.run()
vt = res['vt']['classification']
vt.groupby(['cohort', 'class']).agg(celdas=('cell', 'size'), polizas=('policies', 'sum'), prima=('annual_premium', 'sum'), fcf=('fcf', 'sum'))

celdas  polizas     prima        fcf
cohort class                                      
2023   ONE        99      429   405,079    166,500
       RES        68      406   855,476   -580,069
       SIG       151     2165 2,069,786 -2,535,350
2024   ONE        99      420   360,226    152,764
       RES        68      424   903,267   -606,553
       SIG       145     2156 2,024,920 -2,332,170
2025   RES        39      271   698,029   -466,307
       SIG       177     2729 2,864,057 -3,783,526

El canal promocional cae casi completo en ONE en 2023 y 2024: el descuento de 20% deja la prima por debajo del valor presente de siniestros, gastos y ajuste por riesgo. Algunas celdas promocionales de 20 años quedan en RES, porque el margen de mejora de mortalidad compensa el descuento en plazos largos. Las edades 50+ caen en RES porque la tarifa les dejó un margen más delgado. En 2025 ya no hay promoción y se retarificó el segmento de suma alta, por lo que no hay grupo oneroso.

In [2]:
vt.pivot_table(index='cell', columns='class', values='policies', aggfunc='sum', fill_value=0).head(27)

class,ONE,RES,SIG
cell,,,
VT|T10|25-39|agentes,0,0,750
VT|T10|25-39|bancaseguros,0,0,280
VT|T10|25-39|promo,105,0,0
VT|T10|40-49|agentes,0,0,769
VT|T10|40-49|bancaseguros,0,0,250
VT|T10|40-49|promo,111,0,0
VT|T10|50-60|agentes,0,349,0
VT|T10|50-60|bancaseguros,0,116,0
VT|T10|50-60|promo,45,0,0


## Daños: el indicador de onerosidad del PAA

En el PAA se presume que no hay contratos onerosos salvo que los hechos indiquen lo contrario (par. 18). El indicador es el ratio combinado esperado con ajuste por riesgo: hasta 95% es SIG, entre 95% y 100% es RES y por encima de 100% es ONE (propuesta del memo). Flotas comerciales queda en RES en 2023 y 2024, y en 2025, con la inflación de repuestos ya conocida, nace onerosa.

In [3]:
res['paa']['classification'][['line', 'segment', 'cohort', 'premium', 'loss_ratio', 'combined_ratio', 'class', 'group']]

,line,segment,cohort,premium,loss_ratio,combined_ratio,class,group
0,AUTO,flotas,2023,"1,355,160",0.6702,0.9629,RES,AUTO-2023-RES
1,AUTO,particulares,2023,"10,851,928",0.6384,0.9016,SIG,AUTO-2023-SIG
2,AUTO,flotas,2024,"1,379,616",0.6770,0.9704,RES,AUTO-2024-RES
3,AUTO,particulares,2024,"11,088,704",0.6423,0.9057,SIG,AUTO-2024-SIG
4,AUTO,flotas,2025,"1,484,538",0.7378,1.0369,ONE,AUTO-2025-ONE
5,AUTO,particulares,2025,"11,330,430",0.6473,0.9109,SIG,AUTO-2025-SIG
6,INC,hogar_comercio,2023,"3,993,772",0.5110,0.9153,SIG,INC-2023-SIG
7,INC,hogar_comercio,2024,"4,068,924",0.5151,0.9200,SIG,INC-2024-SIG
8,INC,hogar_comercio,2025,"4,154,980",0.5190,0.9245,SIG,INC-2025-SIG
9,TRC,construccion,2023,"3,463,890",0.5500,0.8429,SIG,TRC-2023-SIG


## Elección del modelo para Todo Riesgo Construcción (par. 53(a) y 54)

Las obras duran tres años, así que el PAA solo cabe si se espera que su medición no difiera materialmente del GMM. El motor proyecta el LRC con ambos modelos en cada cierre de la cobertura y en siete escenarios, que se conocen al final del primer año:

- **Base.**
- **Siniestralidad ±20%:** en el GMM la absorbe el CSM (par. 44(c)).
- **Tasas ±100 pb:** en el GMM es un efecto financiero (B97(a)).
- **Avance de obra más lento o más rápido:** cambia el patrón de unidades de cobertura y el de ingresos del PAA (B126(b)).

El umbral es 5% del LRC del GMM.

In [4]:
tests = pd.concat([pc.trc_eligibility(y) for y in (2023, 2024, 2025)])
tabla = tests[tests['lrc_gmm'] > 0].pivot_table(index=['cohort', 'scenario'], columns='year_end', values='relative')
(tabla * 100).round(2)

year_end              2023    2024    2025    2026    2027
cohort scenario                                           
2023   base         1.3900 -1.1100 -3.4000     NaN     NaN
       claims_+20%  1.3900 -0.8900 -2.9000     NaN     NaN
       claims_-20%  1.3900 -1.3300 -3.8900     NaN     NaN
       rates_+100bp 1.3900 -0.0400 -2.7300     NaN     NaN
       rates_-100bp 1.3900 -2.1700 -4.0700     NaN     NaN
       works_faster 1.3900 -1.2800 -3.7100     NaN     NaN
       works_slower 1.3900 -1.1100 -3.3600     NaN     NaN
2024   base            NaN  1.0400 -1.4500 -3.5500     NaN
       claims_+20%     NaN  1.0400 -1.2100 -3.0100     NaN
       claims_-20%     NaN  1.0400 -1.7000 -4.0900     NaN
       rates_+100bp    NaN  1.0400 -0.4300 -2.8900     NaN
       rates_-100bp    NaN  1.0400 -2.4700 -4.2200     NaN
       works_faster    NaN  1.0400 -1.6400 -3.8700     NaN
       works_slower    NaN  1.0400 -1.4600 -3.5200     NaN
2025   base            NaN     NaN  0.7500 -1.7800 -3.9900
       claims_+20%     NaN     NaN  0.7500 -1.6400 -3.6200
       claims_-20%     NaN     NaN  0.7500 -1.9300 -4.3500
       rates_+100bp    NaN     NaN  0.7500 -0.7500 -3.3300
       rates_-100bp    NaN     NaN  0.7500 -2.8100 -4.6400
       works_faster    NaN     NaN  0.7500 -1.9300 -4.2700
       works_slower    NaN     NaN  0.7500 -1.8200 -3.9800

In [5]:
{y: (pc.trc_eligibility(y).attrs['eligible'], round(100 * pc.trc_eligibility(y).attrs['max_relative'], 2)) for y in (2023, 2024, 2025)}

{2023: (True, 4.07), 2024: (True, 4.22), 2025: (True, 4.64)}

Las tres cohortes son elegibles: la mayor diferencia es 4,6%, en el escenario de tasas −100 pb y al final del segundo año. La diferencia viene de que el PAA no descuenta su LRC (par. 56) mientras el GMM sí, y de que el CSM se acrecienta a tasa bloqueada. La conclusión se documenta por cohorte (par. 97(a)) y TRC se mide con el PAA, con ingresos según el patrón esperado de siniestros (B126(b)).

## Por qué vida con ahorro no es VFA (B101)

La tasa acreditada la declara la aseguradora cada año y el contrato no especifica un conjunto identificado de elementos subyacentes; B106(b) describe exactamente este caso. Falla B101(a), y sin él no pueden cumplirse B101(b) y (c). Vida con ahorro se mide con el GMM; su compromiso de acreditación (tasa de mercado menos 1,25%, con mínimo de 2,5%) se especifica al inicio según B98–B100.